# Analyze Drudge hyperlinks

In [1]:
import re
import sys
from pathlib import Path
from urllib.parse import urlparse

In [2]:
import storysniffer

In [3]:
import numpy as np
import pandas as pd
import altair as alt

In [4]:
this_dir = Path("__file__").parent.absolute()
sys.path.append(this_dir.parent)
sys.path.append(str(this_dir.parent / "newshomepages"))

In [5]:
import altair_theme

In [6]:
alt.themes.register('palewire', altair_theme.theme)
alt.themes.enable('palewire')

ThemeRegistry.enable('palewire')

In [7]:
extracts_dir = this_dir.parent / "extracts" / "csv"

In [8]:
analysis_dir = this_dir.parent / "_analysis"

Read in the sample data

In [9]:
df = pd.read_csv(
    extracts_dir / "drudge-hyperlinks-sample.csv",
    usecols=[
        'handle',
        'file_name',
        'date',
        'text',
        'url',
    ],
    dtype=str,
    parse_dates=["date"]
)

In [10]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 61755 entries, 0 to 61754
Data columns (total 5 columns):
 #   Column     Non-Null Count  Dtype         
---  ------     --------------  -----         
 0   handle     61755 non-null  object        
 1   file_name  61755 non-null  object        
 2   date       61755 non-null  datetime64[ns]
 3   text       61489 non-null  object        
 4   url        61754 non-null  object        
dtypes: datetime64[ns](1), object(4)
memory usage: 2.4+ MB


In [11]:
df.head()

,handle,file_name,date,text,url
0,DRUDGE,drudge-2022-05-31T14:43:07.863919-04:00.hyperl...,2022-05-31,Kremlin propaganda declares 'WW3 has started'...,https://www.dailystar.co.uk/news/world-news/ru...
1,DRUDGE,drudge-2022-05-31T14:43:07.863919-04:00.hyperl...,2022-05-31,Threatens to wipe out USA with just four Satan...,https://www.the-sun.com/news/5460070/russia-th...
2,DRUDGE,drudge-2022-05-31T14:43:07.863919-04:00.hyperl...,2022-05-31,Moscow Military Brass Caught Venting: 'You're ...,https://www.thedailybeast.com/top-russian-mili...
3,DRUDGE,drudge-2022-05-31T14:43:07.863919-04:00.hyperl...,2022-05-31,Zelensky warns of famine risk from blockade of...,https://thehill.com/policy/international/russi...
4,DRUDGE,drudge-2022-05-31T14:43:07.863919-04:00.hyperl...,2022-05-31,Cracks Show in Western Front...,https://www.wsj.com/articles/cracks-appear-in-...


Guess links with `storysniffer`

In [12]:
links_df = df.groupby(["text", "url"]).size().rename("n").reset_index()

In [13]:
sniffer = storysniffer.StorySniffer()

In [14]:
links_df.describe()

,n
count,7103.000000
mean,8.656624
std,39.302088
min,1.000000
25%,1.000000
50%,2.000000
75%,3.000000
max,254.000000


In [15]:
links_df['is_story'] = links_df.apply(lambda x: sniffer.guess(x['url'], text=x['text']), axis=1)

In [16]:
links_df.is_story.value_counts()

True     6835
False     268
Name: is_story, dtype: int64

In [17]:
links_df.is_story.value_counts(normalize=True)

True     0.962269
False    0.037731
Name: is_story, dtype: float64

In [18]:
links_df.to_csv("./drudge-hyperlinks-storysniffer-guesses.csv", index=False)

Make some manual fixes

In [19]:
blacklist = [
    "/privacy/",
]

In [20]:
links_df.loc[
    links_df.url.isin(blacklist),
    'is_story'
] = False

In [21]:
substack_pattern = ".(substack|theankler|commonsense).(com|news)/p/"

In [22]:
links_df.loc[
    links_df.url.str.contains(substack_pattern),
    'is_story'
] = True

/tmp/ipykernel_1839511/673093257.py:2: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  links_df.url.str.contains(substack_pattern),


In [23]:
time_pattern = "^https://time.com/\d{5,}/*"

In [24]:
links_df.loc[
    links_df.url.str.contains(time_pattern),
    'is_story'
] = True

In [25]:
studyfinds_pattern = "^https://www.studyfinds.org/*.{5,}"

In [26]:
links_df.loc[
    links_df.url.str.contains(studyfinds_pattern),
    'is_story'
] = True

In [27]:
bbc_pattern = "^https://www.bbc.com/news/*.{5,}"

In [28]:
links_df.loc[
    links_df.url.str.contains(bbc_pattern),
    'is_story'
] = True

In [29]:
jpost_pattern = "^https://www.jpost.com/breaking-news/*.{5,}"

In [30]:
links_df.loc[
    links_df.url.str.contains(jpost_pattern),
    'is_story'
] = True

In [31]:
braint_pattern = "^https://www.braintomorrow.com/*.{5,}"

In [32]:
links_df.loc[
    links_df.url.str.contains(braint_pattern),
    'is_story'
] = True

Knock out anything that appears most of the time

In [33]:
n = len(df.file_name.unique())

In [42]:
links_df.loc[
    links_df.n >= n * .5,
    'is_story',
] = False

In [43]:
links_df.to_csv("./drudge-hyperlinks-storysniffer-tweaks.csv", index=False)

In [44]:
links_df[
    (links_df.n < 40) &
    (links_df.is_story == False)
].sort_values("url").head(5)

,text,url,n,is_story,domain
3468,LIVE HEAT MAP...,http://hp2.wright-weather.com/icons/us_heat.gif,8,False,hp2.wright-weather.com
3473,LIVE: TEMP MAP,http://hp2.wright-weather.com/icons/us_heat.gif,2,False,hp2.wright-weather.com
3474,LIVE: TEMP MAP...,http://hp2.wright-weather.com/icons/us_heat.gif,14,False,hp2.wright-weather.com
4505,PARTICIPANTS...,https://bilderbergmeetings.org/press/press-rel...,1,False,bilderbergmeetings.org
1691,DEPP JUSTICE!,https://deadline.com/tag/depp-heard-trial/,1,False,deadline.com


Tally domains

In [45]:
links_df['domain'] = links_df.url.apply(lambda x: urlparse(x).netloc)

In [46]:
links_df[links_df.is_story].domain.value_counts().reset_index().head(10)

,index,domain
0,www.msn.com,810
1,apnews.com,503
2,www.wsj.com,377
3,news.yahoo.com,323
4,dnyuz.com,301
5,www.dailymail.co.uk,295
6,www.cnbc.com,219
7,www.the-sun.com,217
8,www.cnn.com,210
9,nypost.com,202


In [53]:
story_df = links_df[links_df.is_story].copy()

In [81]:
def is_trump(text):
    token_list = [t.lower() for t in text.split()]
    if 'trump' in token_list:
        return True
    elif 'donald' in token_list:
        return True
    elif 'don' in token_list:
        return True
    else:
        return False

In [82]:
story_df['is_trump'] = story_df.text.apply(is_trump)

In [83]:
story_df.is_trump.value_counts()

False    6678
True      178
Name: is_trump, dtype: int64

In [84]:
trump_df = story_df[story_df.is_trump].copy()

In [85]:
date_df = df[['url', 'date']].sort_values(["url", "date"], ascending=[True, True]).drop_duplicates("url", keep="first")

In [86]:
trump_df = trump_df.merge(date_df, on="url").sort_values("date", ascending=True)

In [89]:
trump_df.tail()

,text,url,n,is_story,domain,is_trump,date
69,NY AG's investigation into Trump nears endgame...,https://www.vanityfair.com/news/2022/08/letiti...,1,True,www.vanityfair.com,True,2022-08-26
68,NY AG probe into Trump nears endgame...,https://www.vanityfair.com/news/2022/08/letiti...,3,True,www.vanityfair.com,True,2022-08-26
114,The Don Rages At FBI 'Hacks and Thugs'...,https://www.mediaite.com/trump/trump-attacks-d...,6,True,www.mediaite.com,True,2022-08-26
82,"Of all legal threats, is this one that could t...",https://www.theguardian.com/us-news/2022/aug/2...,5,True,www.theguardian.com,True,2022-08-27
28,Don Jr. Posts Meme About His Fathers Crotch...,https://www.thedailybeast.com/donald-trump-jr-...,5,True,www.thedailybeast.com,True,2022-08-27


In [88]:
list(trump_df.text.unique())

['Herschel Walker Accuses Trump of Lying About Convincing Him to Run...',
 "'Outsider' in NV Senate primary surges, rattling Trump pick...",
 "Republicans who texted Meadows say The Don could've stopped violence...",
 'Trump election probe grand jury to hear from Raffensperger...',
 'Former Trump aide Peter Navarro indicted for contempt of Congress...',
 'Trump Summer Plans Include 7-Hour Grilling in Fraud Suit...',
 'The Don weighs big bet in Alabama Senate race...',
 "Trump call to 'walk down to Capitol' prompted Secret Service scramble...",
 "TRUMP ISSUES STATEMENT:  'DRUDGE IS DEAD'...",
 'Biden Trashes Trump On Kimmel...',
 'Will Put The Don at Center of Plot...',
 'NYPOST:  Republicans should leave Trump -- and obsessed Dems -- behind...',
 'The Don Throws Her Under the Bus...',
 "Trump at Center of 'Attempted Coup'...",
 "Mo Brooks Lashes Out at 'Conned' Trump After Rival Gets Endorsement...",
 "NO MORE IN '24:  GERGEN SAYS BIDEN, TRUMP TOO OLD...",
 "Panel hears: Trump 'detache